# 09 — Stemming
**Goal:** Understand rule-based word reduction.

Stemming reduces words by stripping suffixes with rules — no dictionary, no part-of-speech awareness. Porter (1980) and Snowball (Porter2) are the classic algorithms, and they are extremely fast: a pure string transform applicable to any vocabulary in any domain. The cost of that speed is crudeness — stems are frequently not real words.

**Why it matters for resumes / ATS:** stemming is a legitimate choice for high-volume search indexing where recall-per-byte beats precision. But resume–JD matching is exact-keyword matching on real words, and a stem like `engin` never appears in a job description. This chapter builds the case that Ch. 08's lemmatization — not stemming — belongs in the resume pipeline.

| Algorithm | Speed | Accuracy | Best For |
|---|---|---|---|
| Porter | Fast | Lower | Search indexing, TF-IDF |
| Snowball | Fast | Slightly better | General-purpose stemming |
| Lemmatization (Ch. 08) | Slower | High | Resume matching, NER |

## 1. Porter vs Snowball

Porter and Snowball are sibling rule sets: Snowball (Porter2) is a cleaner, better-documented rewrite of Porter with improved rules for common English endings. On most words they agree; the differences show up on edge cases.

| Input | Porter | Snowball | Issue |
|---|---|---|---|
| `running` | `run` | `run` | ✓ Suffix stripped correctly |
| `ran` | `ran` | `ran` | ❌ Irregular — not recognized |
| `better` | `better` | `better` | ❌ Not mapped to `good` |
| `happily` | `happili` | `happili` | ❌ Non-word result |
| `programming` | `program` | `program` | ✓ Works |
| `analyses` | `analys` | `analys` | ❌ Not `analysis` |
| `leaves` | `leav` | `leav` | ❌ Not `leaf` |

In [ ]:
from nltk.stem import PorterStemmer, SnowballStemmer

p, s = PorterStemmer(), SnowballStemmer("english")

print(f"{'Input':<15} {'Porter':<12} {'Snowball':<12} {'Real Word?'}")
print("-" * 55)
for w in ["running", "ran", "better", "happily", "programming", "analyses", "leaves"]:
    porter = p.stem(w)
    snowball = s.stem(w)
    is_word = porter.isalpha() and len(porter) > 2  # rough check
    print(f"{w:<15} {porter:<12} {snowball:<12} {'✓' if is_word and porter not in ('happili','analys','leav','ran','better') else '✗'}")

**Key observations:**
- `ran` → `ran`: no rule covers irregulars; stemming cannot know `ran` is `run`
- `better` → `better`: not `good` — no vocabulary lookup
- `happili` → non-word; the `ily` ending is chopped but no dictionary repairs it

## 2. The Crudeness Problem

Aggressive suffix stripping over-conflates: words that should stay distinct collapse onto the same stem, and technical vocabulary gets mangled past recognition.

| Pair | Porter Stem | Problem |
|---|---|---|
| `university` / `universal` | `univers` / `univers` | School = adjective? ❌ |
| `organization` / `organize` | `organ` / `organ` | Abstract = action? ❌ |
| `engineering` / `engineer` | `engin` / `engin` | Stem is not a word ❌ |

In [ ]:
print(f"{'Input':<15} {'Stem':<10} {'Same Stem?'}")
print("-" * 40)
pairs = [
    ("university", "universal"),
    ("organization", "organize"),
    ("engineering", "engineer"),
]
for w1, w2 in pairs:
    s1, s2 = PorterStemmer().stem(w1), PorterStemmer().stem(w2)
    same = "= SAME" if s1 == s2 else ""
    print(f"{w1:<15} {s1:<10}")
    print(f"{w2:<15} {s2:<10} {same}")
    print()

**Try it:** these pairs are the argument in miniature: any exact match against a JD fails, and the false conflations merge concepts an ATS must keep separate. When your stems stop looking like words, the algorithm has stopped serving the task.

## Decision Guide

| Use Case | Recommended Approach | Why |
|---|---|---|
| Fast search indexing | Stemming | Speed, TF-IDF vocab reduction |
| Resume–JD matching | Lemmatization | Exact keyword matching required |
| NER / entity extraction | Lemmatization | POS-aware, real words |
| High-volume filtering | Stemming | Recall-per-byte wins |

For this project the decision is made: lemmatize (Ch. 08) and preserve proper nouns. Stemming remains the fallback for languages or domains without a lemmatizer — and it closes the word-form trilogy (Ch. 07 stop words → Ch. 08 lemmas → Ch. 09 stems).

## Key Insight

**Stemming is fast but crude — it produces non-words that break exact matching.**

Porter and Snowball are pure string transforms: no vocabulary, no POS, no irregular forms. They work for TF-IDF and search indexing, but resume–JD matching requires real words. The stems `engin`, `organ`, and `univers` never appear in job descriptions.

The next chapter, Ch. 10 POS tagging, explains the signal that makes lemmatization accurate — and why stemming can never match it.